# Phase 4.1 - Aggressive Model Compression

**Author:** Rafael  **Dataset:** Tiny-ImageNet-200 (200 classes, 64x64 RGB)

Push the best Phase 1-3 models below INT8. PyTorch-native quantization
strategies, evaluated on the same axis so bit-widths are directly comparable:

| Method | Bits | Mode | Models |
|---|---|---|---|
| INT8 | W8/A8 | PTQ (calibrate) | all 5 (anchor) |
| INT4 PTQ | W4/A8 | PTQ (calibrate) | all 5 |
| INT4 QAT | W4/A8 | retrain | 3 tiny |
| Mixed | W4-8/A8 per-layer | retrain | 3 tiny |
| INT2 PTQ | W2/A8 | PTQ (calibrate) | all 5 |
| Ternary (TWN) | `{-α,0,+α}` weight-only | PTQ (no calib) | all 5 |
| Binary (BWN) | `{-α,+α}` weight-only | PTQ (no calib) | all 5 |

INT4/mixed/INT2 use **FakeQuantize simulation** (fbgemm has no INT4 kernel): accuracy is
faithful, size is the theoretical packed size (`theoretical_size_mb`). Ternary/binary
quantize weights directly (`apply_weight_ptq`), activations FP32. All eval on GPU.

**Targets:** `mobilenetv2`, `alexnet_bottleneck`, `alexnet_fire`,
`alexnet_depthwisesep` (QAT-unstable edge case), `alexnet_final_fire_residual`.

Layout: 1. Imports 2. Data 3. FP32 baselines 4. Compression helpers
5. INT8 6. INT4 PTQ 7. INT4 QAT 8. Mixed precision 8b. Sub-INT4 (INT2/ternary/binary) 9. Comparison & summaries

## 1. Imports & reproducibility

In [ ]:
import json
import os
import random
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchinfo
import wandb

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    Trainer, make_qat_callback, auto_resume_path,
    load_best_model, build_comparison_table, create_results_summary,
    disk_mb, compute_flops,
    make_qconfig, prepare_sim, calibrate,
    compute_layer_sensitivity, assign_mixed_precision, apply_weight_ptq, theoretical_size_mb,
)
from ml import find_fuse_groups
from configs.loader import load_config

from models import (
    MobileNetV2TV, AlexNetBottleneck, AlexNetFire,
    AlexNetDepthwiseSep, AlexNetFinalFireResidual,
)

torch.backends.quantized.engine = "fbgemm"

In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR    = project_root / "checkpoints" / "compression_phase4_1"
RESULTS_DIR = project_root / "results" / "compression_phase4_1"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data_cfg = DataConfig(**load_config("data.yaml"))
data_cfg.seed = SEED
fp32_cfg = TrainerConfig(**load_config("training.yaml"))
comp_cfg = load_config("compression.yaml")
print(device)

cuda


## 2. Dataset & loaders

Standard train/val loaders plus a small **calibration loader** (single-worker, no shuffle) used for PTQ range collection and sensitivity analysis.

In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path

train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg)

calib_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True,
)

print(f"Train {len(train_ds):,} | Val {len(val_ds):,} | classes {data_cfg.num_classes}")
print(f"Batches: train={len(train_loader)} val={len(val_loader)}")

Train 90,000 | Val 10,000 | classes 200
Batches: train=1406 val=157


## 3. Target models & FP32 baselines

The 5 targets live in three different Phase 1-3 checkpoint dirs. We load each
FP32 `*_best.pth`, evaluate it, and record its baseline (accuracy, disk size, params, MACs).

In [4]:
TARGETS = {
    "mobilenetv2":                 dict(ctor=MobileNetV2TV,            src="baselines_qat_phase1"),
    "alexnet_bottleneck":          dict(ctor=AlexNetBottleneck,        src="compensation_phase3"),
    "alexnet_fire":                dict(ctor=AlexNetFire,              src="compensation_phase3"),
    "alexnet_depthwisesep":        dict(ctor=AlexNetDepthwiseSep,      src="compensation_phase3"),
    "alexnet_final_fire_residual": dict(ctor=AlexNetFinalFireResidual, src="final_architecture_phase4"),
}
ALL_MODELS  = list(TARGETS)
TINY_MODELS = ["alexnet_bottleneck", "alexnet_fire", "alexnet_depthwisesep"]

IMG = data_cfg.img_size
INPUT_SIZE = (1, 3, IMG, IMG)


def load_fp32(name):
    """Load the best FP32 checkpoint for a target from its source phase dir."""
    spec = TARGETS[name]
    src_dir = project_root / "checkpoints" / spec["src"]
    return load_best_model(name, spec["ctor"], src_dir, device, eval_mode=True)


def eval_on(model, loader):
    """Top-1/Top-5/loss via Trainer.evaluate (device follows model)."""
    t = Trainer(model, train_loader, loader, replace(fp32_cfg, use_amp=False),
                next(model.parameters()).device, SAVE_DIR, "eval",
                num_classes=data_cfg.num_classes)
    return t.evaluate(topk=(1, 5))

In [5]:
fp32_baseline = {}
for name in ALL_MODELS:
    m = load_fp32(name)
    info = torchinfo.summary(m, input_size=INPUT_SIZE, verbose=0)
    ev = eval_on(m, val_loader)
    flops = compute_flops(m.cpu().eval(), input_size=INPUT_SIZE)
    src_dir = project_root / "checkpoints" / TARGETS[name]["src"]
    fp32_baseline[name] = {
        "top1": ev["top1"], "top5": ev["top5"],
        "params_m": info.total_params / 1e6,
        "disk_mb": disk_mb(src_dir / f"{name}_best.pth"),
        "size_mb_32": theoretical_size_mb(m, w_bits=32),
        "macs": flops["macs"],
    }
    print(f"{name:30s} top1={ev['top1']:.2f}% top5={ev['top5']:.2f}% "
          f"params={info.total_params/1e6:.2f}M size={fp32_baseline[name]['disk_mb']:.1f}MB")
    del m; torch.cuda.empty_cache()

mobilenetv2                    top1=58.00% top5=81.51% params=2.48M size=28.8MB
alexnet_bottleneck             top1=44.62% top5=71.04% params=0.39M size=4.5MB
alexnet_fire                   top1=43.98% top5=70.43% params=0.52M size=6.0MB
alexnet_depthwisesep           top1=44.39% top5=69.45% params=0.31M size=3.7MB
alexnet_final_fire_residual    top1=49.79% top5=74.80% params=0.70M size=8.1MB


## 4. Compression helpers

`fuse_map_for` auto-detects Conv-BN(-ReLU) groups. `run_ptq` calibrates a
fake-quant model (no retraining). `run_qat` fine-tunes one. Every result is
appended to `ROWS` in the shared schema and mirrored to a per-model record.

In [6]:
ROWS = []
per_model = {n: {"fp32_baseline": fp32_baseline[n], "methods": []} for n in ALL_MODELS}


def fuse_map_for(name):
    """Best-effort auto fuse map (find_fuse_groups recurses into .features etc.)."""
    return find_fuse_groups(TARGETS[name]["ctor"]())


def _load_best_into(model, run_name):
    ckpt = torch.load(SAVE_DIR / f"{run_name}_best.pth", map_location=str(device), weights_only=False)
    model.load_state_dict(ckpt.get("model_state_dict", ckpt))
    return model


def record(name, method, precision, ev, size_mb, bench, notes,
           train_hrs=None, calib_min=None, bits_map=None):
    base = fp32_baseline[name]
    ratio = base["size_mb_32"] / size_mb if size_mb else None
    row = {
        "model": name, "method": method, "precision": precision,
        "fp32_baseline_acc": base["top1"], "compressed_top1_acc": ev["top1"],
        "top5_acc": ev["top5"], "acc_drop_pp": base["top1"] - ev["top1"],
        "compression_ratio": ratio,
        "fp32_size_mb": base["size_mb_32"], "compressed_size_mb": size_mb,
        "latency_ms": bench["latency_ms_per_image"],
        "throughput_img_s": bench["throughput_img_per_s"],
        "params_m": base["params_m"], "acc_per_mb": ev["top1"] / size_mb if size_mb else None,
        "training_time_hrs": train_hrs, "calibration_time_min": calib_min,
        "top1_%": ev["top1"], "top5_%": ev["top5"], "size_MB": size_mb,
        "notes": notes,
    }
    ROWS.append(row)
    rec = dict(row); rec["bits_map"] = bits_map
    per_model[name]["methods"].append(rec)
    print(f"  -> {method:12s} top1={ev['top1']:.2f}% ({row['acc_drop_pp']:+.2f}pp) "
          f"size={size_mb:.2f}MB ratio={ratio:.1f}x")


def wandb_run(name, method, cfg_extra):
    return wandb.init(
        project="alexnet-compression", group="phase-4-1-compression",
        name=f"{name}_{method}",
        config={"arch": name, "method": method, "phase": "4.1",
                "dataset": "tiny-imagenet-200", **cfg_extra},
        tags=["phase-4-1", method, name, "tiny-imagenet", "compression"],
        mode="offline", reinit=True,
    )

In [7]:
def run_ptq(name, method, w_bits, a_bits, n_calib, precision, notes):
    """Post-training: fuse -> fake-quant -> calibrate -> eval. No retraining."""
    model = load_fp32(name)
    qconfig = make_qconfig(w_bits=w_bits, a_bits=a_bits, per_channel=True)
    sim = prepare_sim(model, fuse_map_for(name), qconfig, device)
    t0 = time.perf_counter()
    sim = calibrate(sim, calib_loader, device, n_samples=n_calib)
    calib_min = (time.perf_counter() - t0) / 60
    ev = eval_on(sim, val_loader)
    bench = Trainer(sim, train_loader, val_loader, replace(fp32_cfg, use_amp=False),
                    device, SAVE_DIR, "bench", num_classes=data_cfg.num_classes).benchmark(warmup=50)
    size_mb = theoretical_size_mb(model, w_bits=w_bits)
    run = wandb_run(name, method, {"w_bits": w_bits, "a_bits": a_bits, "calib_samples": n_calib})
    run.summary.update({"top1": ev["top1"], "top5": ev["top5"], "size_mb": size_mb})
    wandb.finish()
    record(name, method, precision, ev, size_mb, bench, notes, calib_min=calib_min)
    del model, sim; torch.cuda.empty_cache()


def run_qat(name, method, w_bits, a_bits, mcfg, precision, notes, bits_map=None, sim=None):
    """QAT: fine-tune a fake-quant (or mixed) model, reload best, eval."""
    model = load_fp32(name)
    if sim is None:
        qconfig = make_qconfig(w_bits=w_bits, a_bits=a_bits, per_channel=mcfg.get("per_channel", True))
        sim = prepare_sim(model, fuse_map_for(name), qconfig, device)
    run_name = f"{method}_{name}"
    qat_cfg = replace(fp32_cfg, epochs=mcfg["epochs"], lr=mcfg["lr"],
                      weight_decay=mcfg["weight_decay"], use_amp=False)
    cb = make_qat_callback(mcfg["freeze_bn_epoch"], mcfg["disable_observer_epoch"])
    resume = auto_resume_path(SAVE_DIR, run_name)

    if (SAVE_DIR / f"{run_name}_best.pth").exists() and not resume:
        print(f"  (reusing {run_name}_best.pth)")
    else:
        run = wandb_run(name, method, {"w_bits": w_bits, "a_bits": a_bits, **mcfg})
        tr = Trainer(sim, train_loader, val_loader, qat_cfg, device, SAVE_DIR, run_name,
                     num_classes=data_cfg.num_classes, wandb_run=run, epoch_callback=cb,
                     log_file=SAVE_DIR / f"{run_name}.log")
        fit = tr.fit(resume_from=resume)
        wandb.finish()
        per_model[name].setdefault("train_time_hrs", {})[method] = fit["total_training_time_s"] / 3600

    sim = _load_best_into(sim, run_name).eval()
    ev = eval_on(sim, val_loader)
    bench = Trainer(sim, train_loader, val_loader, qat_cfg, device, SAVE_DIR, "bench",
                    num_classes=data_cfg.num_classes).benchmark(warmup=50)
    size_mb = theoretical_size_mb(model, w_bits=w_bits, bits_map=bits_map)
    train_hrs = per_model[name].get("train_time_hrs", {}).get(method)
    record(name, method, precision, ev, size_mb, bench, notes,
           train_hrs=train_hrs, bits_map=bits_map)
    del model, sim; torch.cuda.empty_cache()

## 5. INT8 (anchor)

Uniform fake-quant W8/A8 PTQ across all 5 models - the known-good reference point. (Real INT8-QAT numbers from Phases 1-3 remain valid cross-checks.)

In [8]:
for name in ALL_MODELS:
    print(name)
    run_ptq(name, "int8", w_bits=8, a_bits=8,
            n_calib=comp_cfg["int4_ptq"]["calibration_samples"],
            precision="int8", notes="fake-quant W8/A8 PTQ")

mobilenetv2


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


size_mb,2.36518
top1,55.41303
top5,79.93391


  -> int8         top1=55.41% (+2.58pp) size=2.37MB ratio=4.0x
alexnet_bottleneck


size_mb,0.36716
top1,44.23525
top5,70.65533


...

  -> int8         top1=43.94% (+0.04pp) size=0.49MB ratio=4.0x
alexnet_depthwisesep


size_mb,0.29911
top1,43.65471
top5,69.48452


  -> int8         top1=43.65% (+0.74pp) size=0.30MB ratio=4.0x
alexnet_final_fire_residual


size_mb,0.6663
top1,49.75736
top5,74.84932


  -> int8         top1=49.76% (+0.04pp) size=0.67MB ratio=4.0x


## 6. INT4 PTQ

Weights to 4-bit post-training (per-channel), activations INT8. The fast low-bit baseline.

In [9]:
for name in ALL_MODELS:
    print(name)
    run_ptq(name, "int4_ptq", w_bits=4, a_bits=8,
            n_calib=comp_cfg["int4_ptq"]["calibration_samples"],
            precision="int4", notes="fake-quant W4/A8 PTQ, per-channel")

mobilenetv2


size_mb,1.19895
top1,3.5026
top5,11.59575


  -> int4_ptq     top1=3.50% (+54.49pp) size=1.20MB ratio=7.8x
alexnet_bottleneck


size_mb,0.18536
top1,35.32445
top5,62.18264


  -> int4_ptq     top1=35.32% (+9.30pp) size=0.19MB ratio=7.9x
alexnet_fire


...

  -> int4_ptq     top1=37.72% (+6.26pp) size=0.25MB ratio=7.9x
alexnet_depthwisesep


size_mb,0.15161
top1,9.99649
top5,25.41479


  -> int4_ptq     top1=10.00% (+34.39pp) size=0.15MB ratio=7.8x
alexnet_final_fire_residual


size_mb,0.3354
top1,40.01107
top5,66.9007


  -> int4_ptq     top1=40.01% (+9.78pp) size=0.34MB ratio=7.9x


## 7. INT4 QAT

Retrain under 4-bit weight fake-quant on the 3 tiny models - does training recover the PTQ gap? (Watch **alexnet_depthwisesep**, the -2.92 pp QAT edge case.)

In [10]:
mcfg_int4 = comp_cfg["int4_qat"]
for name in TINY_MODELS:
    print(name)
    run_qat(name, "int4_qat", w_bits=4, a_bits=8, mcfg=mcfg_int4,
            precision="int4", notes="fake-quant W4/A8 QAT, per-channel")

alexnet_bottleneck


Validation: 100%|██████████| 157/157 [00:01<00:00, 102.44it/s, loss=3.0773, top1=40.63%, top5=67.50%]
Epoch  19/20 | train_loss=3.0436 train_acc=40.91% | val_loss=3.0773 val_acc=40.63% val_top5=67.50% | lr=6.16e-08 peak_mem=487MB time=32.8s
Early stopping at epoch 19

================= Run Summary =================
Model          : int4_qat_alexnet_bottleneck
Epochs         : 19
Best Val Top-1 : 44.46%
Best Val Top-5 : 70.71%
Final Val Top-1: 40.63%
Final Val Top-5: 67.50%
Best Val Loss  : inf
Total Time     : 00:00:32


epoch_time_s,▁
lr,▁
peak_gpu_mem_mb,▁
train_acc,▁
train_loss,▁
val_acc,▁
val_loss,▁
val_top5,▁
epoch_time_s,32.82947
lr,0.0
peak_gpu_mem_mb,486.74902


  -> int4_qat     top1=44.34% (+0.29pp) size=0.19MB ratio=7.9x
alexnet_fire


Validation: 100%|██████████| 157/157 [00:02<00:00, 57.13it/s, loss=2.9910, top1=44.41%, top5=70.60%]
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch  11/20 | train_loss=3.0449 train_acc=41.67% | val_loss=2.9910 val_acc=44.41% val_top5=70.60% | lr=4.22e-06 peak_mem=1057MB time=67.1s
Validation: 100%|██████████| 157/157 [00:02<00:00, 52.73it/s, loss=2.9895, top1=44.36%, top5=70.38%]
Epoch  12/20 | train_loss=3.0359 train_acc=41.92% | val_loss=2.9895 val_acc=44.36% val_top5=70.38% | lr=3.45e-06 peak_mem=1057MB time=68.0s
Validation: 100%|██████████| 157/157 [00:02<00:00, 53.55it/s, loss=3.0479, top1=42.49%, top5=69.14%]
Epoch  13/20 | train_loss=3.0359 train_acc=42.01% | val_loss=3.0479 val_acc=42.49% val_top5=69.14% | lr=2.73e-06 peak_mem=1057MB time=68.2s
Valid

best_val_acc,▁
epoch_time_s,▁▄▅▇▇█
lr,█▆▅▃▂▁
peak_gpu_mem_mb,▁▁▁▁▁▁
train_acc,▁▄▆█▅▇
train_loss,█▃▃▁▂▂
val_acc,██▁▃▇▃
val_loss,▁▁█▇▁▇
val_top5,█▇▁▃▇▁
best_val_acc,44.41
epoch_time_s,69.08963


  -> int4_qat     top1=44.26% (-0.28pp) size=0.25MB ratio=7.9x
alexnet_depthwisesep


Validation: 100%|██████████| 157/157 [00:02<00:00, 71.35it/s, loss=3.4601, top1=33.76%, top5=59.25%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/20 | train_loss=3.2564 train_acc=36.47% | val_loss=3.4601 val_acc=33.76% val_top5=59.25% | lr=9.94e-06 peak_mem=340MB time=34.1s
Validation: 100%|██████████| 157/157 [00:02<00:00, 75.07it/s, loss=3.5440, top1=31.75%, top5=57.19%]
Epoch   2/20 | train_loss=3.2209 train_acc=37.35% | val_loss=3.5440 val_acc=31.75% val_top5=57.19% | lr=9.76e-06 peak_mem=340MB time=32.3s
Validation: 100%|██████████| 157/157 [00:02<00:00, 62.10it/s, loss=3.7329, top1=28.17%, top5=52.49%]
Epoch   3/20 | train_loss=3.2226 train_acc=37.29% | val_loss=3.7329 val_acc=28.17% val_top5=52.49% | lr=9.46e-06 peak_mem=340MB time=33.8s
Validation: 100%|██████████| 157/157 [00:02<00:00, 66.80it/s, loss=3.1761, top1=39.45%, top5=65.44%]
Epoch   4/20 | train_loss=3.3048 train_acc=35.44% | val_loss=3.1761 val_acc=39

best_val_acc,▁▆▇██
epoch_time_s,▇▆▇█▅▂▂▂▂▂▃▂▁▁▂▂
lr,███▇▇▆▆▅▅▄▄▃▂▂▁▁
peak_gpu_mem_mb,███▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▄▇▇▁▅▇▆▇▇▇▆██▇██
train_loss,▅▂▂█▅▃▃▃▂▂▂▁▂▂▁▁
val_acc,▄▃▁▇▇▇▇█▇██▆██▇█
val_loss,▅▆█▂▂▂▂▁▂▁▁▃▁▁▁▁
val_top5,▄▃▁▇▇▇▇█▇██▆████
best_val_acc,41.38
epoch_time_s,27.99425


  -> int4_qat     top1=41.32% (+3.07pp) size=0.15MB ratio=7.8x


## 8. Mixed precision (INT4/INT8 per-layer)

Rank layers by INT4 sensitivity, keep the top `int8_ratio` at INT8 and the rest at INT4, then fine-tune. Tests whether per-layer mixing beats uniform INT4.

In [11]:
import copy as _copy
mcfg_mix = comp_cfg["mixed_precision"]
for name in TINY_MODELS:
    print(name)
    model = load_fp32(name)
    base = _copy.deepcopy(model).to(device).train()
    # sensitivity + per-layer qconfig on the UNFUSED graph so layer names match,
    # THEN fuse (carries each conv's qconfig into the fused module) and prepare_qat.
    sens = compute_layer_sensitivity(base, calib_loader, device, w_bits=4)
    bits_map = assign_mixed_precision(base, sens, int8_ratio=mcfg_mix["int8_ratio"],
                                      per_channel=mcfg_mix["per_channel"])
    base.qconfig = make_qconfig(8, 8, mcfg_mix["per_channel"])  # default for unmarked mods
    fm = fuse_map_for(name)
    if fm:
        try:
            torch.ao.quantization.fuse_modules_qat(base, fm, inplace=True)
        except Exception as e:
            print("  fusion skipped:", e)
    base.train()  # fuse_modules_qat flips to eval; prepare_qat requires train mode
    torch.ao.quantization.prepare_qat(base, inplace=True)
    n8 = sum(v == 8 for v in bits_map.values()); n4 = sum(v == 4 for v in bits_map.values())
    print(f"  layers: INT8={n8} INT4={n4}")
    run_qat(name, "mixed", w_bits=4, a_bits=8, mcfg=mcfg_mix,
            precision="int4/8", notes=f"per-layer INT4/INT8 ({n8} hi / {n4} lo)",
            bits_map=bits_map, sim=base)
    del model, base; torch.cuda.empty_cache()

alexnet_bottleneck
  layers: INT8=5 INT4=11


Validation: 100%|██████████| 157/157 [00:02<00:00, 54.21it/s, loss=2.9547, top1=44.38%, top5=70.78%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/20 | train_loss=3.0129 train_acc=41.45% | val_loss=2.9547 val_acc=44.38% val_top5=70.78% | lr=9.94e-06 peak_mem=498MB time=45.3s
Validation: 100%|██████████| 157/157 [00:02<00:00, 56.42it/s, loss=2.9494, top1=44.77%, top5=70.69%]
Epoch   2/20 | train_loss=2.9912 train_acc=42.11% | val_loss=2.9494 val_acc=44.77% val_top5=70.69% | lr=9.76e-06 peak_mem=498MB time=44.5s
Validation: 100%|██████████| 157/157 [00:02<00:00, 58.12it/s, loss=2.9669, top1=43.88%, top5=70.31%]
Epoch   3/20 | train_loss=2.9900 train_acc=42.30% | val_loss=2.9669 val_acc=43.88% val_top5=70.31% | lr=9.46e-06 peak_mem=498MB time=44.9s
Validation: 100%|██████████| 157/157 [00:03<00:00, 48.92it/s, loss=2.9502, top1=44.50%, top5=70.69%]
Epoch   4/20 | train_loss=2.9332 train_acc=43.55% | val_loss=2.9502 val_acc=44

best_val_acc,▁█
epoch_time_s,▇▇▇██▂▁
lr,██▇▆▄▃▁
peak_gpu_mem_mb,███▁▁▁▁
train_acc,▁▃▃▇█▇█
train_loss,█▆▆▂▁▁▁
val_acc,▅█▁▆▅▇▄
val_loss,▄▂█▂▁▁▅
val_top5,▆▅▁▅█▅▃
best_val_acc,44.77
epoch_time_s,34.90469


  -> mixed        top1=44.61% (+0.01pp) size=0.21MB ratio=6.9x
alexnet_fire
  layers: INT8=5 INT4=11


Validation: 100%|██████████| 157/157 [00:03<00:00, 42.50it/s, loss=2.9775, top1=44.63%, top5=71.12%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/20 | train_loss=3.0349 train_acc=41.90% | val_loss=2.9775 val_acc=44.63% val_top5=71.12% | lr=9.94e-06 peak_mem=1068MB time=76.6s
Validation: 100%|██████████| 157/157 [00:03<00:00, 42.75it/s, loss=2.9723, top1=44.73%, top5=71.00%]
Epoch   2/20 | train_loss=3.0069 train_acc=42.70% | val_loss=2.9723 val_acc=44.73% val_top5=71.00% | lr=9.76e-06 peak_mem=1068MB time=77.8s
Validation: 100%|██████████| 157/157 [00:03<00:00, 41.62it/s, loss=2.9776, top1=44.79%, top5=70.95%]
Epoch   3/20 | train_loss=3.0064 train_acc=42.70% | val_loss=2.9776 val_acc=44.79% val_top5=70.95% | lr=9.46e-06 peak_mem=1068MB time=79.5s
Validation: 100%|██████████| 157/157 [00:03<00:00, 39.56it/s, loss=2.9779, top1=44.58%, top5=71.06%]
Epoch   4/20 | train_loss=2.9469 train_acc=44.18% | val_loss=2.9779 val_acc

best_val_acc,▁▅█
epoch_time_s,▅▆▆▆█▁▂▁
lr,██▇▆▅▄▂▁
peak_gpu_mem_mb,███▁▁▁▁▁
train_acc,▁▃▃▇█▇██
train_loss,█▆▆▂▁▂▁▁
val_acc,▄▆█▃▄▂▁▆
val_loss,▇▃▇▇█▆▆▁
val_top5,▃▂▁▂▄▄▅█
best_val_acc,44.79
epoch_time_s,66.78627


  -> mixed        top1=44.70% (-0.72pp) size=0.29MB ratio=6.7x
alexnet_depthwisesep
  layers: INT8=3 INT4=8


Validation: 100%|██████████| 157/157 [00:02<00:00, 71.20it/s, loss=3.0289, top1=43.89%, top5=69.27%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/20 | train_loss=3.0988 train_acc=40.22% | val_loss=3.0289 val_acc=43.89% val_top5=69.27% | lr=9.94e-06 peak_mem=351MB time=33.5s
Validation: 100%|██████████| 157/157 [00:02<00:00, 66.74it/s, loss=3.0232, top1=43.82%, top5=69.28%]
Epoch   2/20 | train_loss=3.0941 train_acc=40.48% | val_loss=3.0232 val_acc=43.82% val_top5=69.28% | lr=9.76e-06 peak_mem=351MB time=35.4s
Validation: 100%|██████████| 157/157 [00:02<00:00, 66.11it/s, loss=3.0217, top1=43.82%, top5=69.15%]
Epoch   3/20 | train_loss=3.0955 train_acc=40.30% | val_loss=3.0217 val_acc=43.82% val_top5=69.15% | lr=9.46e-06 peak_mem=351MB time=40.2s
Validation: 100%|██████████| 157/157 [00:02<00:00, 56.08it/s, loss=3.0179, top1=43.92%, top5=69.17%]
Epoch   4/20 | train_loss=3.0373 train_acc=41.95% | val_loss=3.0179 val_acc=43

best_val_acc,▁▁▃▃▄▅▆█
epoch_time_s,▄▅█▇▇▄▂▂▃▂▂▂▂▂▁▁▂▄▂▂
lr,███▇▇▇▆▆▅▅▄▃▃▂▂▂▁▁▁▁
peak_gpu_mem_mb,███▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▂▁▇▆▇▆▆▇▇▇▇▇▇█▇▇█▇▇
train_loss,███▃▂▂▂▂▂▁▂▂▂▂▁▂▁▁▂▁
val_acc,▂▁▁▂▂▃▄▄▄▁▅▆▅▄█▄▃▄▅▆
val_loss,█▆▆▄▃▄▃▃▃▃▂▃▂▂▂▁▂▁▁▂
val_top5,▃▃▁▁▄▇▄▁▄▄▅▅█▅▆▄▇▄▇▅
best_val_acc,44.35
epoch_time_s,30.17667


  -> mixed        top1=44.27% (+0.12pp) size=0.16MB ratio=7.5x


## 8b. Sub-INT4: INT2, ternary (TWN) & binary (BWN)

How far can PTQ be pushed? Three ever-more-aggressive weight schemes across all 5 models, **no retraining**:

| Method | Weight levels | Bits | Activations |
|---|---|---|---|
| INT2 PTQ | 4 uniform | W2 | A8 (fake-quant) |
| Ternary (TWN) | `{-α, 0, +α}` | ~2 | FP32 |
| Binary (BWN) | `{-α, +α}` | 1 | FP32 |

- **INT2** rides the existing fake-quant path (`run_ptq`, W2/A8).
- **Ternary** (Ternary Weight Networks): per-channel threshold `Δ = 0.7·mean|w|`, scale `α = mean|w|` over survivors — this is the "(-1, 0, 1)" scheme.
- **Binary** (XNOR-Net BWN): `w → sign(w)·α`, `α = mean|w|` per channel.

Ternary/binary are **weight-only** (`apply_weight_ptq`), the setup those papers report. Expect steep drops without QAT — that is the finding: aggressive PTQ collapses these tiny models, mirroring Section 6's INT4 PTQ gap.

In [ ]:
def run_weight_ptq(name, method, scheme, w_bits_size, precision, notes):
    """Weight-only binary/ternary PTQ (BWN/TWN). Activations FP32; no retraining."""
    model = load_fp32(name)
    q = apply_weight_ptq(model, scheme).to(device).eval()
    ev = eval_on(q, val_loader)
    bench = Trainer(q, train_loader, val_loader, replace(fp32_cfg, use_amp=False),
                    device, SAVE_DIR, "bench", num_classes=data_cfg.num_classes).benchmark(warmup=50)
    size_mb = theoretical_size_mb(model, w_bits=w_bits_size)  # ternary needs 2 bits packed naively
    run = wandb_run(name, method, {"scheme": scheme, "w_bits": w_bits_size, "a_bits": 32})
    run.summary.update({"top1": ev["top1"], "top5": ev["top5"], "size_mb": size_mb})
    wandb.finish()
    record(name, method, precision, ev, size_mb, bench, notes)
    del model, q; torch.cuda.empty_cache()


for name in ALL_MODELS:
    print(name)
    run_ptq(name, "int2_ptq", w_bits=2, a_bits=8,
            n_calib=comp_cfg["int4_ptq"]["calibration_samples"],
            precision="int2", notes="fake-quant W2/A8 PTQ, per-channel")
    run_weight_ptq(name, "ternary_ptq", "ternary", w_bits_size=2,
                   precision="ternary", notes="weight-only TWN {-a,0,+a}, per-channel, A=FP32")
    run_weight_ptq(name, "binary_ptq", "binary", w_bits_size=1,
                   precision="binary", notes="weight-only BWN {-a,+a}, per-channel, A=FP32")

## 9. Comparison table, per-model summaries & experiment summary

In [12]:
df = build_comparison_table(ROWS)
cols = ["model","method","precision","fp32_baseline_acc","compressed_top1_acc","acc_drop_pp",
        "compression_ratio","compressed_size_mb","latency_ms","acc_per_mb","notes"]
df_out = df[[c for c in cols if c in df.columns]].copy()
df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)

by_method = (df.groupby("method")
               .agg(mean_top1=("compressed_top1_acc","mean"),
                    mean_drop_pp=("acc_drop_pp","mean"),
                    mean_ratio=("compression_ratio","mean"),
                    mean_acc_per_mb=("acc_per_mb","mean"),
                    n=("model","count"))
               .reset_index())
by_method.to_csv(RESULTS_DIR / "compression_by_method.csv", index=False)

for name, rec in per_model.items():
    with open(RESULTS_DIR / f"{name}_compression_summary.json", "w") as f:
        json.dump(rec, f, indent=2, default=str)

df_out

,model,method,precision,fp32_baseline_acc,compressed_top1_acc,acc_drop_pp,compression_ratio,compressed_size_mb,latency_ms,acc_per_mb,notes
0,alexnet_bottleneck,int4_qat,int4,44.622979,44.337183,0.285795,7.866022,0.185356,0.127377,239.199970,"fake-quant W4/A8 QAT, per-channel"
1,alexnet_fire,int4_qat,int4,43.976474,44.255564,-0.279090,7.916954,0.247589,0.250961,178.746001,"fake-quant W4/A8 QAT, per-channel"
2,alexnet_depthwisesep,int4_qat,int4,44.390011,41.324723,3.065288,7.810570,0.151607,0.121139,272.577868,"fake-quant W4/A8 QAT, per-channel"
3,alexnet_final_fire_residual,int4_ptq,int4,49.793291,40.011069,9.782222,7.906212,0.335396,0.114304,119.295077,"fake-quant W4/A8 PTQ, per-channel"
4,alexnet_fire,int4_ptq,int4,43.976474,37.718636,6.257838,7.916954,0.247589,0.238250,152.343678,"fake-quant W4/A8 PTQ, per-channel"
5,alexnet_bottleneck,int4_ptq,int4,44.622979,35.324445,9.298533,7.866022,0.185356,0.118632,190.576073,"fake-quant W4/A8 PTQ, per-channel"
6,alexnet_depthwisesep,int4_ptq,int4,44.390011,9.996489,34.393522,7.810570,0.151607,0.086644,65.936839,"fake-quant W4/A8 PTQ, per-channel"
7,mobilenetv2,int4_ptq,int4,57.995582,3.502605,54.492977,7.808952,1.198952,0.282382,2.921389,"fake-quant W4/A8 PTQ, per-channel"
8,alexnet_fire,mixed,int4/8,43.976474,44.696173,-0.719699,6.706719,0.292267,0.385857,152.929331,per-layer INT4/INT8 (5 hi / 11 lo)
9,alexnet_bottleneck,mixed,int4/8,44.622979,44.609120,0.013858,6.946494,0.209892,0.236102,212.533410,per-layer INT4/INT8 (5 hi / 11 lo)


In [13]:
def dominated(r, others):
    for o in others:
        if (o["compressed_top1_acc"] >= r["compressed_top1_acc"]
                and o["compressed_size_mb"] <= r["compressed_size_mb"]
                and o["latency_ms"] <= r["latency_ms"]
                and (o["compressed_top1_acc"] > r["compressed_top1_acc"]
                     or o["compressed_size_mb"] < r["compressed_size_mb"]
                     or o["latency_ms"] < r["latency_ms"])):
            return True
    return False

recs = df.to_dict("records")
pareto = [r for r in recs if not dominated(r, recs)]
pareto_df = pd.DataFrame(pareto).sort_values("compressed_top1_acc", ascending=False)
pareto_df[[c for c in cols if c in pareto_df.columns]].to_csv(RESULTS_DIR / "pareto_frontier.csv", index=False)
print(f"Pareto-optimal: {len(pareto)} of {len(recs)}")
pareto_df[["model","method","compressed_top1_acc","compressed_size_mb","latency_ms"]]

Pareto-optimal: 9 of 16


,model,method,compressed_top1_acc,compressed_size_mb,latency_ms
6,mobilenetv2,int8,55.413032,2.365181,0.289456
7,alexnet_final_fire_residual,int8,49.757361,0.666298,0.124363
3,alexnet_fire,mixed,44.696173,0.292267,0.385857
4,alexnet_bottleneck,mixed,44.609120,0.209892,0.236102
0,alexnet_bottleneck,int4_qat,44.337183,0.185356,0.127377
5,alexnet_depthwisesep,mixed,44.269145,0.157833,0.113626
8,alexnet_depthwisesep,int8,43.654713,0.299111,0.101362
1,alexnet_depthwisesep,int4_qat,41.324723,0.151607,0.121139
2,alexnet_depthwisesep,int4_ptq,9.996489,0.151607,0.086644


In [ ]:
create_results_summary(
    results={
        "phase": "4.1", "title": "Aggressive Model Compression",
        "models": ALL_MODELS,
        "methods": ["int8", "int4_ptq", "int4_qat", "mixed", "int2_ptq", "ternary_ptq", "binary_ptq"],
        "fp32_baseline": fp32_baseline,
        "rows": ROWS,
        "by_method": by_method.to_dict("records"),
        "pareto": pareto_df[["model","method","compressed_top1_acc","compressed_size_mb"]].to_dict("records"),
    },
    config=fp32_cfg,
    output_path=RESULTS_DIR / "experiment_summary.json",
)
print("Saved:", RESULTS_DIR / "experiment_summary.json")

## W&B - syncing offline runs

All runs logged offline under project **`alexnet-compression`**, group
**`phase-4-1-compression`**, phase tag `4.1`. Sync when ready:

```bash
wandb sync --sync-all
```